In [1]:
import torch

data = torch.load(
    "../data/rnn_data.pt",
    weights_only=False
)

word_to_idx = data["word_to_idx"]
vocab_size = data["vocab_size"]
max_length = data["max_length"]

print("Vocabulary size:", vocab_size)
print("Maximum sequence length:", max_length)

Vocabulary size: 124455
Maximum sequence length: 100


In [2]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence


class SentimentRNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_size=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        x = self.embedding(x)

        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed_x)

        hidden = hidden[-1]

        output = self.fc(hidden)

        return output

In [3]:
model = SentimentRNN(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_size=128
)

In [4]:
model.load_state_dict(
    torch.load(
        "../models/sentiment_rnn.pth",
        weights_only=True
    )
)

model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [5]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Stopwords
stop_words = set(stopwords.words("english"))

# Preserve sentiment-important negation words
negation_words = {
    "not", "no", "nor", "never", "neither", "n't"
}

stop_words -= negation_words

# Stemmer
ps = PorterStemmer()

In [6]:
def preprocess_for_prediction(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r"http\S+", "", text)

    # 3. Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # 4. Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # 5. Tokenization
    tokens = word_tokenize(text)

    # 6. Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 7. Stemming
    tokens = [
        ps.stem(word)
        for word in tokens
    ]

    # 8. Convert tokens to vocabulary IDs
    token_ids = [
        word_to_idx.get(
            token,
            word_to_idx["<UNK>"]
        )
        for token in tokens
    ]

    # 9. Truncate
    token_ids = token_ids[:max_length]

    # 10. Actual sequence length
    length = len(token_ids)

    # Avoid zero-length sequence
    if length == 0:
        raise ValueError(
            "The review contains no usable tokens after preprocessing."
        )

    # 11. Padding
    token_ids += [
        word_to_idx["<PAD>"]
    ] * (max_length - length)

    return token_ids, length

In [7]:
def predict_sentiment(text):

    # Convert review into token IDs and get actual length
    token_ids, length = preprocess_for_prediction(text)

    # Add batch dimension
    X = torch.tensor(
        [token_ids],
        dtype=torch.long
    )

    # Length also needs a batch dimension
    lengths = torch.tensor(
        [length],
        dtype=torch.long
    )

    # Evaluation mode
    model.eval()

    # No gradients needed during prediction
    with torch.no_grad():

        # RNN produces a logit
        output = model(X, lengths)

        # Convert logit → probability
        probability = torch.sigmoid(output).item()

    # Classification
    sentiment = (
        "Positive"
        if probability >= 0.5
        else "Negative"
    )

    return sentiment, probability

In [8]:
text = "This movie was absolutely amazing. I really loved it!"

sentiment, probability = predict_sentiment(text)

print("Review:", text)
print("Sentiment:", sentiment)
print("Probability:", round(probability, 4))

Review: This movie was absolutely amazing. I really loved it!
Sentiment: Positive
Probability: 0.9427


In [9]:
reviews = [
    "This movie was fantastic. I loved every minute of it!",
    "This was the worst movie I have ever watched.",
    "The movie was okay, nothing special.",
    "Amazing acting and a great story.",
    "I really hated this movie.",
    "I did not enjoy this movie at all."
]

for review in reviews:
    sentiment, probability = predict_sentiment(review)

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print("Probability:", round(probability, 4))


Review: This movie was fantastic. I loved every minute of it!
Sentiment: Positive
Probability: 0.9218

Review: This was the worst movie I have ever watched.
Sentiment: Negative
Probability: 0.0258

Review: The movie was okay, nothing special.
Sentiment: Negative
Probability: 0.0124

Review: Amazing acting and a great story.
Sentiment: Positive
Probability: 0.985

Review: I really hated this movie.
Sentiment: Negative
Probability: 0.1356

Review: I did not enjoy this movie at all.
Sentiment: Negative
Probability: 0.1886
